In [2]:
"""
Phase 1 & Phase 2 — Significance Tests (Paired per-arm controls)
Each treatment arm tested against its OWN per-arm control group.
"""
import warnings
import pandas as pd
import statsmodels.api as sm
warnings.simplefilter("ignore")

PHASE_1_INPUT = "Data/performance_experimental_phase_21_2026-03-13.csv"
PHASE_1_OUTPUT = "Output/phase1_significance_results_2026-03-13.xlsx"

PHASE_2_INPUT = "Data/Output_phase_2/performance_experimental_phase_2_2026-04-17.csv"
PHASE_2_OUTPUT = "Output/phase2_significance_results_2026-04-17.xlsx"

PHASE_3_INPUT = "Data/Output_phase_2/performance_experimental_phase_2_2026-06-16.csv"
PHASE_3_OUTPUT = "Output/phase2_significance_results_lasting_profit_2026-06-16.xlsx"

def load_data(path: str, phase: int = 2) -> pd.DataFrame:
    df = pd.read_csv(path)
    if phase == 1:
        # Pattern: BNLX_ChurnP_{incentive}_{controle|test}_export.csv
        parts = df["churn_group"].str.extract(
            r"BNLX_ChurnP_(?P<incentive>\w+?)_(?P<role>controle|test)_export\.csv"
        )
        df["incentive"] = parts["incentive"]
        df["is_control"] = parts["role"] == "controle"
        df["base_model"] = "ChurnP"
    else:
        split = df["churn_group"].str.split(" - ", n=1, expand=True)
        raw_model = split[0].str.strip()
        df["incentive"] = split[1].str.strip()
        df["is_control"] = raw_model.str.contains("control", case=False)
        df["base_model"] = raw_model.str.replace(r"\s*control\s*:?\s*", "", regex=True).str.strip()
    df["reactivated"] = (df["reactivated"] > 0).astype(int)
    return df

def _one_sided(model_cls, y_ctrl, y_trt):
    y = pd.concat([y_ctrl, y_trt])
    x = sm.add_constant([0] * len(y_ctrl) + [1] * len(y_trt))
    mod = model_cls(y, x).fit(disp=0) if model_cls == sm.Logit else model_cls(y, x).fit()
    coef, p2 = mod.params.iloc[1], mod.pvalues.iloc[1]
    p = p2 / 2 if coef > 0 else 1.0
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    return coef, round(p, 4), stars


def _build_row(model, inc, c, t):
    _, p_r, s_r = _one_sided(sm.Logit, c["reactivated"], t["reactivated"])
    _, p_s, s_s = _one_sided(sm.OLS, c["total_sales"], t["total_sales"])
    _, p_m, s_m = _one_sided(sm.OLS, c["margin"], t["margin"])
    return {
        "Model": model, "Incentive": inc,
        "N_Ctrl": len(c), "N_Treat": len(t),
        "React_Ctrl": round(c["reactivated"].mean(), 4),
        "React_Treat": round(t["reactivated"].mean(), 4),
        "Uplift_React": round(t["reactivated"].mean() - c["reactivated"].mean(), 4),
        "p_react": p_r, "sig_react": s_r,
        "Sales_Ctrl": round(c["total_sales"].mean(), 4),
        "Sales_Treat": round(t["total_sales"].mean(), 4),
        "Uplift_Sales": round(t["total_sales"].mean() - c["total_sales"].mean(), 4),
        "p_sales": p_s, "sig_sales": s_s,
        "Margin_Ctrl": round(c["margin"].mean(), 4),
        "Margin_Treat": round(t["margin"].mean(), 4),
        "Uplift_Margin": round(t["margin"].mean() - c["margin"].mean(), 4),
        "p_margin": p_m, "sig_margin": s_m,
    }


def test_pooled(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model, mdf in df.groupby("base_model"):
        c, t = mdf[mdf["is_control"]], mdf[~mdf["is_control"]]
        if c.empty or t.empty:
            continue
        rows.append(_build_row(model, "ALL (pooled)", c, t))
    return pd.DataFrame(rows).sort_values("p_react").reset_index(drop=True)


def test_per_arm(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, inc), gdf in df.groupby(["base_model", "incentive"]):
        c, t = gdf[gdf["is_control"]], gdf[~gdf["is_control"]]
        if c.empty or t.empty:
            continue
        rows.append(_build_row(model, inc, c, t))
    return pd.DataFrame(rows).sort_values(["Model", "p_react"]).reset_index(drop=True)


def run_phase(input_path: str, output_path: str, label: str, phase: int):
    df = load_data(input_path, phase=phase)
    print(f"\n{'='*60}")
    print(f"{label}: Loaded {len(df):,} rows | Models: {df['base_model'].unique().tolist()}")
    pooled = test_pooled(df)
    per_arm = test_per_arm(df)
    with pd.ExcelWriter(output_path, engine="openpyxl") as w:
        pooled.to_excel(w, sheet_name="Pooled", index=False)
        per_arm.to_excel(w, sheet_name="Per Arm", index=False)
    print(f"\n=== {label} POOLED ===")
    print(pooled.to_string(index=False))
    print(f"\n=== {label} PER ARM ===")
    print(per_arm.to_string(index=False))
    print(f"\nSaved to {output_path}")


if __name__ == "__main__":
    run_phase(PHASE_1_INPUT, PHASE_1_OUTPUT, "PHASE Exp 1", phase=1)
    run_phase(PHASE_2_INPUT, PHASE_2_OUTPUT, "PHASE Exp 2", phase=2)
    run_phase(PHASE_3_INPUT, PHASE_3_OUTPUT, "PHASE Exp 3", phase=2)
    


PHASE Exp 1: Loaded 268,142 rows | Models: ['ChurnP']

=== PHASE Exp 1 POOLED ===
 Model    Incentive  N_Ctrl  N_Treat  React_Ctrl  React_Treat  Uplift_React  p_react sig_react  Sales_Ctrl  Sales_Treat  Uplift_Sales  p_sales sig_sales  Margin_Ctrl  Margin_Treat  Uplift_Margin  p_margin sig_margin
ChurnP ALL (pooled)  134852   133290      0.0207       0.0229        0.0022      0.0       ***      0.5478       0.6272        0.0794   0.0001       ***       0.3402        0.3857         0.0455    0.0002        ***

=== PHASE Exp 1 PER ARM ===
 Model Incentive  N_Ctrl  N_Treat  React_Ctrl  React_Treat  Uplift_React  p_react sig_react  Sales_Ctrl  Sales_Treat  Uplift_Sales  p_sales sig_sales  Margin_Ctrl  Margin_Treat  Uplift_Margin  p_margin sig_margin
ChurnP       5eu   14934    14830      0.0205       0.0247        0.0043   0.0068        **      0.5458       0.6897        0.1439   0.0103         *       0.3436        0.4134         0.0698    0.0352          *
ChurnP       250   15013    15